# Train a smaller NN-duel forecaster on non-overlapping histories

Run **14_prepare_nonoverlap_nn_duel_skill_forecasting.ipynb** first. This notebook ingests its
verified dataset and trains new neural networks using the same training machinery
as **09b_paired_skill_forecasting.ipynb**, with the expanded input projection and **width 96 / dropout 0.25**.
Notebook 14 supplies training windows with disjoint 56-frame raw input footprints,
including across the two views of each flight; origins are at least six seconds
apart. Test and validation keep the dense evaluation windows. The loader rejects
notebook 12's overlapping-training dataset, even if supplied by path override.
For A observing B, inputs contain **A's 54 observation/acceleration features plus
A's current catalogue-skill indicators**; all three output heads predict **B's future
skill**, at **+1, +2 and +5 seconds**. Both directions share the same model.

**Run All trains three independent networks (seeds 42, 43 and 44) by default**, each
for up to 100 epochs with early stopping active from epoch one. Change `TRAINING_SEEDS` to
run fewer or more replicas. Architecture: two Transformer layers, width 96,
six attention heads, dropout 0.25, AdamW, masked equal-horizon cross-entropy,
L1 regularisation and gradient clipping. A single network has three horizon heads, rather than three
independent horizon models. No results or trained weights are bundled here.

Each run uses a fresh timestamped folder and preserves every strict improvement in
mean **test** NLL. As in 09b, **test is the model-selection partition**; validation
is held out until training and checkpoint selection have finished for every seed.
Validation results are descriptive and must not be used to select seeds or tune
settings. Catalogue selections are maneuver labels, not proof of distinct physical execution.

Install `.[ml]` and a Jupyter/IPython kernel first. CUDA uses BF16 when supported;
CPU uses FP32. The default streaming data path avoids caching the approximately
13.7 GiB full feature array on a 16 GiB GPU. Full training can take substantial
time and disk space. Every code cell is timed; GPU work is synchronized.

For a Windows PyArrow import failure, see the [runtime repair instructions](../docs/nn_duel_skill_forecasting.md#windows-pyarrow-import-failure).

This experiment has separate data receipts, run directories and an MLflow experiment.
Use `BVR_NN_NONOVERLAP_DATA` for a prepared-data override and
`BVR_NN_NONOVERLAP_MLFLOW=0` to disable tracking. Fresh runs are required when the
architecture or dataset changes. The smaller training set may omit some rare labels;
inspect notebook 14's retained support report before interpreting results.


In [1]:
%%time
from pathlib import Path
import os
import sys

root_override = os.environ.get("BVR_NN_ROOT")
candidates = ([Path(root_override).expanduser()] if root_override else
              [Path.cwd(), *Path.cwd().parents])
ROOT = next((p.resolve() for p in candidates
             if (p / "bvr_behavior_prediction/nn_duel_skill_forecasting.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Set BVR_NN_ROOT to the full Trajectory_Classification checkout path.")
sys.path.insert(0, str(ROOT))
from bvr_behavior_prediction.notebook_runtime import require_pyarrow
require_pyarrow()  # Report the kernel environment and native-library repair steps.
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from bvr_behavior_prediction import nn_duel_skill_forecasting as duel_data
from bvr_behavior_prediction.paired_skill_forecasting import (
    HORIZONS, HISTORY_STEPS, RAW_HISTORY_STEPS, SPLITS, read_json, write_json, sha256,
)
EXPERIMENT_ROOT = ROOT / "artifacts/nn_duel_nonoverlap_forecasting"
EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)

import torch
from bvr_behavior_prediction.paired_skill_forecasting import (
    DEFAULT_TRAINING, train_model, final_evaluation, metric_tables, resolve_precision,
)
from bvr_behavior_prediction.paired_skill_tracking import log_run_artifacts

DATA_OVERRIDE = os.environ.get("BVR_NN_NONOVERLAP_DATA")
if not DATA_OVERRIDE and not (EXPERIMENT_ROOT / "latest_dataset.json").exists():
    raise FileNotFoundError("Run notebook 14 first, or set BVR_NN_NONOVERLAP_DATA to its dataset directory.")
receipt = None if DATA_OVERRIDE else read_json(EXPERIMENT_ROOT / "latest_dataset.json")
DATA = Path(DATA_OVERRIDE or receipt["directory"]).expanduser().resolve()
if receipt and sha256(DATA / "manifest.json") != receipt["manifest_sha256"]:
    raise ValueError("The prepared dataset changed since notebook 14 completed.")
data = duel_data.NonoverlappingNNDuelData(DATA)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TRAINING_SEEDS = [42, 43, 44]  # One independently fitted three-horizon NN per seed.
TRAIN_MODEL = True
RESUME = False
EXISTING_RUNS = {}  # For resume/report-only: {42: r"C:\...\runs\nn-duel-nonoverlap-s42_...", ...}.
if not TRAINING_SEEDS or len(set(TRAINING_SEEDS)) != len(TRAINING_SEEDS):
    raise ValueError("Choose at least one distinct training seed.")
if (RESUME or not TRAIN_MODEL) and set(EXISTING_RUNS) != set(TRAINING_SEEDS):
    raise ValueError("Set EXISTING_RUNS for every selected seed when resuming or reporting.")
RUN_STAMP = pd.Timestamp.now(tz="UTC").strftime("%Y%m%dT%H%M%S%fZ")
RUNS = {seed: Path(EXISTING_RUNS[seed]).expanduser().resolve() if seed in EXISTING_RUNS
        else EXPERIMENT_ROOT / "runs" / f"nn-duel-nonoverlap-s{seed}_{RUN_STAMP}"
        for seed in TRAINING_SEEDS}
CONFIG = dict(DEFAULT_TRAINING)
CONFIG.update(width=96, dropout=0.25, epochs=100, early_stopping=True, early_stopping_min_epochs=0,
              early_stopping_patience=10, l1_lambda=1e-6,
              overtraining_shutdown=True, overtraining_patience=3,
              overtraining_relative_increase=0.05, overtraining_min_gap=0.1,
              precision="bf16", batch_size=512,
              evaluation_batch_size=1024, data_backend="prefetch" if DEVICE == "cuda" else "direct",
              num_workers=2, cpu_threads=4, compile_model=False)
MLFLOW_ENABLED = os.environ.get("BVR_NN_NONOVERLAP_MLFLOW", "1") == "1"
MLFLOW_TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI",
    f"sqlite:///{(EXPERIMENT_ROOT / 'mlflow.db').as_posix()}")
TRACKING = {"enabled": MLFLOW_ENABLED, "tracking_uri": MLFLOW_TRACKING_URI,
            "experiment": "nn-duel-nonoverlap-skill-forecasting", "log_per_class": True,
            "log_model": True,
            "artifact_root": (EXPERIMENT_ROOT / "mlflow_artifacts").as_uri()
                             if MLFLOW_TRACKING_URI.startswith("sqlite:") else None}
print("Prepared dataset:", DATA)
print("Model run directories:", {s: str(p) for s, p in RUNS.items()})
display(pd.Series({**CONFIG, "device": DEVICE,
    "effective_precision": resolve_precision(DEVICE, CONFIG["precision"]),
    "input_features": len(data.manifest["features"]), "classes": len(data.classes),
    "history_steps": HISTORY_STEPS, "horizons_s": HORIZONS}))
display(pd.DataFrame(data.manifest["counts"]).T)
if MLFLOW_ENABLED and MLFLOW_TRACKING_URI.startswith("sqlite:"):
    print(f'mlflow ui --backend-store-uri "{MLFLOW_TRACKING_URI}" --host 127.0.0.1 --port 5000')
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    torch.cuda.synchronize()


Prepared dataset: C:\Users\theon\Trajectory_Classification\artifacts\nn_duel_nonoverlap_forecasting\data\eee3474c5493_5b951842316d_0a5d2704_7661ce18
Model run directories: {42: 'C:\\Users\\theon\\Trajectory_Classification\\artifacts\\nn_duel_nonoverlap_forecasting\\runs\\nn-duel-nonoverlap-s42_20261109T141336690795Z', 43: 'C:\\Users\\theon\\Trajectory_Classification\\artifacts\\nn_duel_nonoverlap_forecasting\\runs\\nn-duel-nonoverlap-s43_20261109T141336690795Z', 44: 'C:\\Users\\theon\\Trajectory_Classification\\artifacts\\nn_duel_nonoverlap_forecasting\\runs\\nn-duel-nonoverlap-s44_20261109T141336690795Z'}


epochs                                  100
seed                                     42
batch_size                              512
evaluation_batch_size                  1024
learning_rate                        0.0003
weight_decay                           0.01
gradient_clip                           1.0
width                                    96
heads                                     6
layers                                    2
dropout                                0.25
early_stopping                         True
early_stopping_min_epochs                 0
early_stopping_patience                  10
precision                              bf16
data_backend                       prefetch
num_workers                               2
prefetch_factor                           2
cpu_threads                               4
loader_timeout                           60
compile_model                         False
compile_mode                        default
l1_lambda                       

,groups,episodes,windows,labels_by_horizon
train,4000,8000,137001,"[137001, 136791, 136135]"
test,3000,6000,1224086,"[1224086, 1218086, 1200100]"
validation,3000,6000,1224414,"[1224414, 1218414, 1200422]"


mlflow ui --backend-store-uri "sqlite:///C:/Users/theon/Trajectory_Classification/artifacts/nn_duel_nonoverlap_forecasting/mlflow.db" --host 127.0.0.1 --port 5000
GPU: NVIDIA GeForce RTX 5080 Laptop GPU
CPU times: total: 23.9 s
Wall time: 23.9 s


## Fit independently seeded networks

The loss averages the three available horizon cross-entropies with independent
missing-label masks. Scaling and the split are fixed by notebook 14 for every seed.
The training objective adds `l1_lambda * sum(abs(weights))` (default `1e-6`) to
masked cross-entropy. This includes trainable matrix/tensor weights and position
embeddings, excluding biases and LayerNorm scales. AdamW weight decay remains 0.01.
Train/test NLL is evaluated without L1, in evaluation mode; checkpoint selection
therefore remains comparable across epochs. History records the L1 contribution,
evaluation-mode training objective and test-minus-train NLL gap separately.

Early-stopping patience counts from epoch one: ten consecutive epochs without a
new all-time minimum test NLL stop the run (earliest plateau stop: epoch 11).
There is no minimum training duration. A separate overtraining shutdown stops after
three consecutive epochs where all of these hold: test NLL exceeds its best by
more than 5%, test-minus-train NLL exceeds 0.1, and train NLL is below its value at
the best-test checkpoint. A recovery resets this counter. These are configurable
starting thresholds, not tuned guarantees. The shutdown acts independently of
ordinary patience and preserves `best.pt`; counters and the stop reason survive
resume. Set both `early_stopping=False` and `overtraining_shutdown=False` to run
the full epoch budget. `RESUME=True` requires the same source, data and configuration.
Checkpoint input dimensions, feature ordering and dataset digest are stored with weights.


In [2]:
%%time
histories = {}
def epoch_output(row, summary, per_class):
    print(f"Epoch {row['epoch']}: train NLL={row['train_nll']:.5f}, "
          f"test NLL={row['test_nll']:.5f}, L1={row['train_l1_penalty']:.5f}, "
          f"gap={row['generalization_gap']:.5f}, "
          f"overtraining streak={row['overtraining_epochs']}, {row['seconds']:.1f}s", flush=True)
    display(summary[["split", "horizon_s", "nll", "accuracy", "macro_f1",
                     "supported_macro_f1", "weighted_f1"]].round(4))
    with pd.option_context("display.max_rows", None):
        display(per_class.pivot(index=["horizon_s", "skill"], columns="split",
                                 values=["class_nll", "f1", "support"]).round(4))

for seed, run in RUNS.items():
    print(f"Training seed {seed}: {run}", flush=True)
    config = {**CONFIG, "seed": seed}
    tracking = {**TRACKING, "run_name": run.name}
    if TRAIN_MODEL:
        histories[seed] = train_model(data, run, config, callback=epoch_output,
                                     device=DEVICE, resume=RESUME, tracking=tracking)
    else:
        histories[seed] = pd.read_csv(run / "history.csv")
    status = read_json(run / "training_status.json")
    if status["status"] not in {"completed", "early_stopped"}:
        raise ValueError(f"Training has not concluded for seed {seed}.")
    if read_json(run / "run.json")["training"] != config:
        raise ValueError(f"Requested configuration differs from saved seed {seed}.")
    display(histories[seed].tail())
if DEVICE == "cuda":
    torch.cuda.synchronize()


Training seed 42: C:\Users\theon\Trajectory_Classification\artifacts\nn_duel_nonoverlap_forecasting\runs\nn-duel-nonoverlap-s42_20261109T141336690795Z


C:\Users\theon\.conda\envs\trajectory-nn-clean\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/11/09 14:13:40 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/09 14:13:40 INFO mlflow.store.db.utils: Updating database tables


MLflow run: d77c242856d34e68b95f1bac998d1811 (nn-duel-nonoverlap-skill-forecasting)
Runtime: cuda, bf16, prefetch, batch 512, evaluation batch 1024, frame cache 0.000 GiB
Epoch 1: train NLL=1.49770, test NLL=1.48347, L1=0.01484, gap=-0.01424, overtraining streak=0, 23.0s


,split,horizon_s,nll,accuracy,macro_f1,supported_macro_f1,weighted_f1
0,train,1,1.5025,0.4562,0.1090,0.1124,0.4307
1,train,2,1.4977,0.4590,0.1132,0.1168,0.4344
2,train,5,1.4929,0.4616,0.1054,0.1122,0.4352
3,test,1,1.4852,0.4616,0.1108,0.1108,0.4360
4,test,2,1.4824,0.4650,0.1146,0.1146,0.4405
5,test,5,1.4827,0.4650,0.1063,0.1063,0.4389


class_nll              f1           support  \
split                              test   train    test   train      test   
horizon_s skill                                                             
1         abort_support          5.8377  5.5571  0.0000  0.0000    1488.0   
          accelerate_to_speed    5.8150  6.4249  0.0000  0.0000       2.0   
          beam_missile_left      5.3525  5.4051  0.0000  0.0000     253.0   
          beam_missile_right     4.5712  4.4239  0.0000  0.0000    1073.0   
          beam_target_left       1.7527  1.7621  0.4147  0.4258   49072.0   
          beam_target_right      2.5132  2.4512  0.1329  0.1436   22525.0   
          climb_to_altitude      5.1791  4.4461  0.0000  0.0000      20.0   
          commit                 6.4285     NaN  0.0000  0.0000       2.0   
          crank_target_left      2.5068  2.3779  0.2159  0.2002   17193.0   
          crank_target_right     1.7626  1.8002  0.3885  0.3893   37913.0   
          decelerate_to_speed    3.7120  3.7796  0.0000  0.0000    3324.0   
          defensive_reversal     4.8848  4.8330  0.0000  0.0000     455.0   
          descend_to_altitude    3.9067  3.9099  0.0000  0.0000    3126.0   
          dive_defense           6.2763  4.8592  0.0000  0.0000       9.0   
          drag_missile           5.4096  5.5090  0.0000  0.0000      39.0   
          extend                 1.7396  1.7879  0.4296  0.3831  100323.0   
          lag_pursuit            1.6009  1.5889  0.3170  0.3444  194366.0   
          last_ditch_break       6.3695  9.0970  0.0000  0.0000      17.0   
          launch                 4.9366  4.7593  0.0000  0.0000    2710.0   
          lead_pursuit           3.4879  3.3960  0.0069  0.0046   19044.0   
          lock_target            5.8014  5.7958  0.0000  0.0000      60.0   
          maintain_heading       5.6711  5.5834  0.0000  0.0000     309.0   
          notch_left             4.6840  4.4804  0.0000  0.0000    1479.0   
          notch_right            5.4929  5.5642  0.0000  0.0000     635.0   
          preemptive_crank       5.2391  5.3839  0.0000  0.0000      40.0   
          pursue_target          2.8333  2.8808  0.0000  0.0000   54139.0   
          recommit               0.9545  0.9657  0.5815  0.5766  347921.0   
          search                 6.1126  8.3572  0.0000  0.0000      52.0   
          secondary_shot         1.2111  1.2370  0.5451  0.5361  104198.0   
          short_range_attack     3.1272  2.9537  0.1288  0.1074    4443.0   
          support_missile        5.5235  5.4439  0.0000  0.0000     981.0   
          turn_cold              1.2128  1.2307  0.4970  0.4859  256315.0   
          turn_to_heading        5.1450  5.0789  0.0000  0.0000     560.0   
2         abort_support          5.8690  5.5925  0.0000  0.0000    1477.0   
          accelerate_to_speed    6.1610  6.4462  0.0000  0.0000       2.0   
          beam_missile_left      5.1045  5.1065  0.0000  0.0000     248.0   
          beam_missile_right     4.4068  4.3274  0.0000  0.0000    1015.0   
          beam_target_left       1.7286  1.7504  0.4417  0.4506   48555.0   
          beam_target_right      2.4889  2.4318  0.1454  0.1571   22040.0   
          climb_to_altitude      5.4497  5.8045  0.0000  0.0000      19.0   
          commit                 6.5245  5.7998  0.0000  0.0000       2.0   
          crank_target_left      2.6116  2.4579  0.2373  0.2288   16215.0   
          crank_target_right     1.7695  1.7957  0.4108  0.4110   37773.0   
          decelerate_to_speed    3.8581  3.8771  0.0000  0.0000    3202.0   
          defensive_reversal     5.0810  5.0173  0.0000  0.0000     434.0   
          descend_to_altitude    3.6638  3.6117  0.0103  0.0202    2988.0   
          dive_defense           4.9731     NaN  0.0000  0.0000       9.0   
          drag_missile           6.3546  6.4564  0.0000  0.0000      39.0   
          extend                 1.7279  1.7703  0.4304  0.3832  100121.0   
          lag_pursuit     

Epoch 2: train NLL=1.40725, test NLL=1.41399, L1=0.01490, gap=0.00675, overtraining streak=0, 18.6s


... (170 outputs left out)

,split,horizon_s,nll,accuracy,macro_f1,supported_macro_f1,weighted_f1
0,train,1,1.3974,0.4797,0.1246,0.1285,0.4580
1,train,2,1.3950,0.4812,0.1245,0.1284,0.4588
2,train,5,1.3959,0.4800,0.1217,0.1296,0.4578
3,test,1,1.4028,0.4760,0.1224,0.1224,0.4540
4,test,2,1.4008,0.4759,0.1221,0.1221,0.4532
5,test,5,1.4028,0.4756,0.1173,0.1173,0.4530


class_nll              f1           support  \
split                              test   train    test   train      test   
horizon_s skill                                                             
1         abort_support          5.5423  5.2392  0.0000  0.0000    1488.0   
          accelerate_to_speed    7.6520  7.0037  0.0000  0.0000       2.0   
          beam_missile_left      5.8697  6.0476  0.0000  0.0000     253.0   
          beam_missile_right     4.4255  4.2262  0.0000  0.0000    1073.0   
          beam_target_left       1.4146  1.4251  0.5085  0.5118   49072.0   
          beam_target_right      2.4310  2.3689  0.2225  0.2232   22525.0   
          climb_to_altitude      6.4130  5.9833  0.0000  0.0000      20.0   
          commit                 7.3668     NaN  0.0000  0.0000       2.0   
          crank_target_left      2.3518  2.1742  0.3322  0.3522   17193.0   
          crank_target_right     1.7570  1.7581  0.4385  0.4479   37913.0   
          decelerate_to_speed    3.2973  3.3066  0.0012  0.0049    3324.0   
          defensive_reversal     5.9426  5.6437  0.0000  0.0000     455.0   
          descend_to_altitude    3.5965  3.3086  0.0108  0.0052    3126.0   
          dive_defense           6.5813  6.7140  0.0000  0.0000       9.0   
          drag_missile           7.0088  6.8754  0.0000  0.0000      39.0   
          extend                 1.5618  1.5601  0.4368  0.4219  100323.0   
          lag_pursuit            1.5435  1.5179  0.3426  0.3616  194366.0   
          last_ditch_break       7.7862  7.1920  0.0000  0.0000      17.0   
          launch                 5.1901  5.1475  0.0000  0.0000    2710.0   
          lead_pursuit           3.2999  3.1304  0.0370  0.0730   19044.0   
          lock_target            6.8230  6.9782  0.0000  0.0000      60.0   
          maintain_heading       6.2798  6.1564  0.0000  0.0000     309.0   
          notch_left             4.5559  4.4012  0.0000  0.0000    1479.0   
          notch_right            5.6062  5.5340  0.0000  0.0000     635.0   
          preemptive_crank       6.4544  7.3558  0.0000  0.0000      40.0   
          pursue_target          2.5419  2.5475  0.0169  0.0096   54139.0   
          recommit               0.9745  0.9742  0.5753  0.5815  347921.0   
          search                 6.7411  7.9172  0.0000  0.0000      52.0   
          secondary_shot         1.3128  1.3081  0.5576  0.5628  104198.0   
          short_range_attack     3.1114  2.9167  0.0390  0.0359    4443.0   
          support_missile        5.3655  5.2899  0.0000  0.0000     981.0   
          turn_cold              1.0281  1.0247  0.5189  0.5206  256315.0   
          turn_to_heading        5.3822  5.4735  0.0000  0.0000     560.0   
2         abort_support          5.5840  5.2847  0.0000  0.0000    1477.0   
          accelerate_to_speed    7.0449  6.6240  0.0000  0.0000       2.0   
          beam_missile_left      6.0528  6.2359  0.0000  0.0000     248.0   
          beam_missile_right     4.1154  3.9930  0.0000  0.0000    1015.0   
          beam_target_left       1.3782  1.3965  0.5225  0.5265   48555.0   
          beam_target_right      2.4304  2.3663  0.2348  0.2370   22040.0   
          climb_to_altitude      6.1275  6.0996  0.0000  0.0000      19.0   
          commit                 6.9428  6.3853  0.0000  0.0000       2.0   
          crank_target_left      2.3946  2.2208  0.3024  0.3186   16215.0   
          crank_target_right     1.7708  1.7643  0.4560  0.4705   37773.0   
          decelerate_to_speed    3.2042  3.2051  0.0062  0.0049    3202.0   
          defensive_reversal     6.0230  5.8812  0.0000  0.0000     434.0   
          descend_to_altitude    3.6458  3.3973  0.0000  0.0000    2988.0   
          dive_defense           6.0890     NaN  0.0000  0.0000       9.0   
          drag_missile           7.1292  6.7922  0.0000  0.0000      39.0   
          extend                 1.5470  1.5479  0.4357  0.4257  100121.0   
          lag_pursuit     

CPU times: total: 42min 2s
Wall time: 39min 57s


KeyboardInterrupt: 

## Final evaluation and a training-frequency baseline

All seeds have finished selection before this cell accesses validation. The selected
checkpoint of each seed is evaluated on train, test and validation, with per-class
NLL/F1/support, confusion matrices and forecast probabilities saved in its run folder.
A Laplace-smoothed training-frequency baseline uses training labels only.
The default report displays the **first configured seed**, chosen before seeing scores;
no winner is selected using validation.


In [ ]:
%%time
all_metrics = []
for seed, run in RUNS.items():
    best_epoch, summary, scores, results = final_evaluation(
        data, run, DEVICE, tracking={**TRACKING, "run_name": run.name})
    all_metrics.append(summary.assign(seed=seed, selected_epoch=best_epoch))
    display(Markdown(f"**Seed {seed}, test-selected epoch {best_epoch}**"))
    display(summary.round(4))
    del results  # Release full prediction arrays between independently fitted models.
comparison = pd.concat(all_metrics, ignore_index=True)
for run in RUNS.values():
    comparison.to_csv(run / "seed_comparison.csv", index=False)

frequency = np.zeros((3, len(data.classes)), np.float32)
for h in range(len(HORIZONS)):
    labels = data.targets[data.indices["train"], h]
    counts = np.bincount(labels[labels >= 0], minlength=len(data.classes))
    frequency[h] = (counts + 1) / (counts.sum() + len(data.classes))
baseline_rows = []
for split in SPLITS:
    labels = data.targets[data.indices[split]]
    probabilities = np.broadcast_to(frequency, (len(labels), *frequency.shape))
    baseline, _, _ = metric_tables(labels, probabilities, data.classes, split)
    baseline_rows.append(baseline)
baseline_metrics = pd.concat(baseline_rows, ignore_index=True)
for run in RUNS.values():
    baseline_metrics.to_csv(run / "training_frequency_baseline.csv", index=False)
display(baseline_metrics.assign(model="Training frequency").round(4))

REPORT_SEED = TRAINING_SEEDS[0]
RUN = RUNS[REPORT_SEED]
history = histories[REPORT_SEED]
best_epoch = read_json(RUN / "selection.json")["epoch"]
final_metrics = pd.read_csv(RUN / "final_metrics.csv")
final_per_class = pd.read_csv(RUN / "final_per_class.csv")
final_results = {}
for split in SPLITS:
    with np.load(RUN / f"final_{split}.npz") as stored:
        final_results[split] = {key: stored[key] for key in stored.files}
if DEVICE == "cuda":
    torch.cuda.synchronize()


## Diagnostics for the preselected report seed

These are the same learning curves as notebook 13, horizon metrics, per-class support/F1,
confusion matrices and reliability plots, using the new own-skill-conditioned
forecasts. Figures use the noninteractive Agg backend and display saved PNG bytes.
Changing `REPORT_SEED` refreshes a report; it must not become validation-based model selection.


In [ ]:
%%time
DIAGNOSTICS = RUN / "diagnostics"
DIAGNOSTICS.mkdir(exist_ok=True)

# Also select Agg here so rerunning this cell fixes an already-running kernel.
from io import BytesIO
from time import perf_counter
import matplotlib
matplotlib.use("Agg", force=True)
import matplotlib.pyplot as plt
from IPython.display import Image, display

print(f"Diagnostics backend: {matplotlib.get_backend()}; output: {DIAGNOSTICS}", flush=True)

def show_figure(fig, name):
    started = perf_counter()
    def stage(message):
        print(f"[{name}, {perf_counter() - started:.2f}s] {message}", flush=True)

    try:
        stage("Laying out figure")
        # tight_layout fits labels inside the fixed canvas. A second tight-bbox
        # save would render again; inherited savefig.bbox settings are overridden.
        with matplotlib.rc_context({"savefig.bbox": None}):
            fig.tight_layout()
            stage("Rendering PNG")
            with BytesIO() as buffer:
                fig.savefig(buffer, format="png", dpi=160)
                png = buffer.getvalue()
        path = DIAGNOSTICS / f"{name}.png"
        stage("Writing PNG")
        path.write_bytes(png)
        stage("Displaying saved PNG inline")
        display(Image(data=png))  # Reuse the render; plt.show() can block on GUI backends.
        stage(f"Done: {path.name}")
    finally:
        plt.close(fig)  # Release large figures even if saving/display fails.

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for split in ["train", "test"]:
    axes[0].plot(history.epoch, history[f"{split}_nll"], label=split)
    axes[1].plot(history.epoch, history[f"{split}_macro_f1"], label=split)
for ax in axes:
    ax.axvline(best_epoch, color="black", linestyle=":", label="Selected epoch")
    ax.set_xlabel("Epoch")
    ax.legend()
axes[0].set(title="Mean horizon loss", ylabel="NLL / cross-entropy")
axes[1].set(title="Mean horizon catalogue macro-F1", ylabel="F1", ylim=(0, 1))
show_figure(fig, "learning_curves")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, measure in zip(axes, ["nll", "accuracy", "macro_f1"]):
    final_metrics.pivot(index="horizon_s", columns="split", values=measure)[list(SPLITS)].plot(
        kind="bar", ax=ax, rot=0)
    ax.set(title=f"Best checkpoint: {measure}", xlabel="Forecast horizon (s)")
    if measure != "nll": ax.set_ylim(0, 1)
show_figure(fig, "final_split_comparison")

# Plotting uses CPU arrays and does not submit CUDA work.


In [ ]:
%%time
# F1 and support for all catalogue labels, split by horizon.
fig, axes = plt.subplots(2, 3, figsize=(20, 14), sharey=True)
for h, horizon in enumerate(HORIZONS):
    subset = final_per_class.loc[final_per_class.horizon_s.eq(horizon)]
    f1 = subset.pivot(index="skill", columns="split", values="f1").reindex(
        index=data.classes, columns=SPLITS)
    counts = subset.pivot(index="skill", columns="split", values="support").reindex(
        index=data.classes, columns=SPLITS)
    image = axes[0, h].imshow(f1, vmin=0, vmax=1, aspect="auto", cmap="viridis")
    count_image = axes[1, h].imshow(np.log1p(counts), aspect="auto", cmap="magma")
    for ax in axes[:, h]:
        ax.set_xticks(range(3), SPLITS)
        ax.set_yticks(range(len(data.classes)), data.classes, fontsize=7)
    axes[0, h].set_title(f"+{horizon}s per-class F1")
    axes[1, h].set_title(f"+{horizon}s log(1 + support)")
    fig.colorbar(image, ax=axes[0, h], fraction=.04)
    fig.colorbar(count_image, ax=axes[1, h], fraction=.04)
show_figure(fig, "per_class_f1_and_support")

fig, axes = plt.subplots(3, 3, figsize=(24, 23))
for row, split in enumerate(SPLITS):
    for h, horizon in enumerate(HORIZONS):
        cm = final_results[split]["confusion"][h]
        totals = cm.sum(1, keepdims=True)
        normalized = np.divide(cm, totals, out=np.zeros_like(cm, float), where=totals > 0)
        ax = axes[row, h]
        ax.imshow(normalized, vmin=0, vmax=1, cmap="Blues")
        ax.set_xticks(range(len(data.classes)), data.classes, rotation=90, fontsize=6)
        ax.set_yticks(range(len(data.classes)), data.classes, fontsize=6)
        ax.set(title=f"{split}, +{horizon}s", xlabel="Predicted skill", ylabel="True skill")
show_figure(fig, "confusion_matrices")

# Plotting uses CPU arrays and does not submit CUDA work.


In [ ]:
%%time
# Probability calibration: each point averages one confidence bin.
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for h, horizon in enumerate(HORIZONS):
    axes[h].plot([0, 1], [0, 1], "k:", label="Perfect calibration")
    for split in SPLITS:
        result = final_results[split]
        valid = result["targets"][:, h] >= 0
        p = result["probabilities"][valid, h]
        y = result["targets"][valid, h]
        confidence, correct = p.max(-1), p.argmax(-1) == y
        bins = np.minimum((confidence * 10).astype(int), 9)
        points = [(confidence[bins == b].mean(), correct[bins == b].mean())
                  for b in range(10) if (bins == b).any()]
        axes[h].plot(*np.asarray(points).T, marker="o", label=split)
    axes[h].set(title=f"+{horizon}s reliability", xlabel="Mean confidence",
                ylabel="Observed accuracy", xlim=(0, 1), ylim=(0, 1))
    axes[h].legend()
show_figure(fig, "reliability")

with pd.option_context("display.max_rows", None):
    display(final_per_class.pivot(index=["horizon_s", "skill"], columns="split",
                                  values=["class_nll", "f1", "support"]).round(4))
print("Saved metrics, predictions, checkpoints and PNG diagnostics to:", RUN)
if MLFLOW_ENABLED:
    log_run_artifacts(RUN)  # Attach final figures and baseline tables to the same run.
if (RUN / "mlflow.json").exists():
    print("MLflow run ID:", read_json(RUN / "mlflow.json")["run_id"])

# Plotting uses CPU arrays and does not submit CUDA work.


The saved `best.pt` contains the expanded input projection, class vocabulary, feature
ordering, three horizon heads, dataset digest and model configuration. For inference,
recreate the **same causal 51-frame inputs**, including the observing aircraft's own
current skill and training-only scaling. Load with
`PairedTransformer(num_classes=len(checkpoint["classes"]), **checkpoint["model_config"])`
and `model.load_state_dict(checkpoint["model_state"])`. Never substitute the opponent's
current skill for the observer's own skill. Source code and provenance accompany every run.
